In [1]:
import nibabel as nib
from nilearn.image import resample_to_img
from nilearn import plotting
from ipywidgets import interact, widgets
import gmsh
from scipy.io import loadmat
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import socket
import os

hostname = socket.gethostname()
p = {}
if hostname == 'syndrome' or hostname == 'zod.psych.nyu.edu' or hostname == 'zod':
    p['datd'] = '/d/DATD/datd/TMS_V3AB'
else:
    p['datd'] = '/Users/mrugankdake/Documents/Clayspace/EEG_TMS/datd/TMS_V3AB'
p['data'] = p['datd'] + '/data'
p['analysis'] = p['datd'] + '/analysis'
p['meta'] = p['analysis'] + '/meta_analysis'
p['df_fname'] = os.path.join(p['meta'], 'master_df.csv')
p['simnibs_path'] = os.path.join(p['datd'], 'SIMNIBS_output')

# Load up summary data
summary_df = pd.read_csv(os.path.join(p['analysis'] + '/V3AB_TMS_meta - Summary.csv'))

All_metadata = {row['Subject ID']: row for _, row in summary_df.iterrows()}
subs = [2, 14]

stim_intensities = []
hemisphere_stimulated = []
for ss in subs:
    stim_intensities.append(All_metadata[ss]['Stim Intensity'])
    hemisphere_stimulated.append(All_metadata[ss]['Hemisphere stimulated'])
stim_dIdt = [round((a / 60) * 87 *1e6) for a in stim_intensities]

In [2]:
# Function to load nifty
def load_nii_file(fpath, T1):
    nii_img = nib.load(fpath)
    if nii_img.shape[0] != 320:
        nii_img = resample_to_img(fpath, T1, interpolation='nearest')
    data = nii_img.get_fdata()
    return data

# Function to compute average from mask
def calculate_average_over_mask(whole_brain, mask):
    masked_data = whole_brain[mask > 0]
    #print(masked_data.shape)
    average_value = masked_data.max() 
    #########print(average_value)
    return average_value

In [4]:
subfoldpath = f"{p['simnibs_path']}/sub19"
t1_path = f"{subfoldpath}/T1.nii"
lhv3ab_path = f"{subfoldpath}/lh.V3AB.nii.gz"
rhv3ab_path = f"{subfoldpath}/rh.V3AB.nii.gz"

t1_data = load_nii_file(t1_path, t1_path)
roi_lhv3ab = load_nii_file(lhv3ab_path, t1_path)
roi_rhv3ab = load_nii_file(rhv3ab_path, t1_path)
roi_total = roi_lhv3ab + roi_rhv3ab
t1_plot = t1_data.copy()

In [5]:
slice_x = 1
slice_y = 1
slice_z = 1
def update_plot(slice_x, slice_y, slice_z):
    plt.figure(figsize=(15, 15))
    # Subplot for slice_x
    plt.subplot(131)
    plt.imshow(t1_plot[:, slice_x, :], cmap='gray', origin='lower')
    plt.imshow(roi_total[:, slice_x, :], cmap='blue', origin='lower')
    plt.title(f'Slice X: {slice_x}')
    plt.axis('off')
    # Subplot for slice_y
    plt.subplot(132)
    plt.imshow(t1_plot[slice_y, :, :], cmap='blue', origin='lower')
    plt.title(f'Slice Y: {slice_y}')
    plt.axis('off')
    # Subplot for slice_z
    plt.subplot(133)
    plt.imshow(t1_plot[:, :, slice_z], cmap='gray', origin='lower')
    plt.title(f'Slice Z: {slice_z}')
    plt.axis('off')
    plt.show()
# Create widgets for each slice
slice_x_widget = widgets.IntSlider(min=0, max=t1_data.shape[0] - 1, value=slice_x, description='Slice X:')
slice_y_widget = widgets.IntSlider(min=0, max=t1_data.shape[1] - 1, value=slice_y, description='Slice Y:')
slice_z_widget = widgets.IntSlider(min=0, max=t1_data.shape[2] - 1, value=slice_z, description='Slice Z:')
# Create an interactive plot
interact(update_plot, slice_x=slice_x_widget, slice_y=slice_y_widget, slice_z=slice_z_widget)

interactive(children=(IntSlider(value=1, description='Slice X:', max=319), IntSlider(value=1, description='Sli…

<function __main__.update_plot(slice_x, slice_y, slice_z)>